# Tidy Data: Principios y Transformación con Pandas

## 🎯 Objetivos de Aprendizaje
- Comprender la filosofía de **Tidy Data** formalizada por Hadley Wickham.
- Diferenciar claramente entre formato ancho (*wide format*) y formato largo (*long format*).
- Dominar las funciones de reestructuración en Pandas: `melt()`, `pivot()`, `pivot_table()`, `stack()` y `unstack()`.
- Identificar y resolver los problemas comunes de datos tabulares desordenados.
- Construir transformaciones limpias y reproducibles en Python 3.12+.

## 🌉 Puente Pedagógico: La Anatomía de los Datos

### ¿Por qué importa Tidy Data?
En ciencia de datos, hasta un 80% del esfuerzo se consume en preparar y organizar datos. Las librerías científicas como Pandas, Seaborn y Scikit-Learn asumen que los datos vienen en una estructura estandarizada para aprovechar la vectorización.

### Analogía
Piensa en una biblioteca:
- **Datos desordenados (Untidy)**: Fichas con información dispersa, totales mezclados con filas individuales o nombres de medicamentos usados como títulos de columna.
- **Datos ordenados (Tidy)**: Un índice estricto donde cada ficha describe una sola entidad y cada casilla mide un atributo específico.

### Reglas Fundamentales (Hadley Wickham)
1. Cada **variable** forma una columna.
2. Cada **observación** forma una fila.
3. Cada tipo de **unidad observacional** forma una tabla independiente.

### Diagrama ASCII: Wide vs Long
```
       [ FORMATO ANCHO (UNTIDY) ]                  [ FORMATO LARGO (TIDY) ]
  +-----------+----------+----------+        +-----------+------------+--------+
  | Paciente  | Medic_A  | Medic_B  |        | Paciente  | Medicacion | Pulso  |
  +-----------+----------+----------+        +-----------+------------+--------+
  | Ricardo   |    67    |    56    | --+    | Ricardo   |  Medic_A   |   67   |
  | Marielena |    80    |    90    |   |    | Ricardo   |  Medic_B   |   56   |
  | Miguel    |    64    |    50    |   +--> | Marielena |  Medic_A   |   80   |
  +-----------+----------+----------+  melt  | Marielena |  Medic_B   |   90   |
                                             | Miguel    |  Medic_A   |   64   |
                                             | Miguel    |  Medic_B   |   50   |
                                             +-----------+------------+--------+
```

In [ ]:
import pandas as pd
import numpy as np

# Datos clínicos en formato ancho
df_clinico = pd.DataFrame({
    "paciente": ["Ricardo", "Marielena", "Miguel"],
    "medicamento_a": [67, 80, 64],
    "medicamento_b": [56, 90, 50]
})

print("--- Tabla Original (Formato Ancho) ---")
display(df_clinico)

## 1. De Ancho a Largo con `pd.melt()`

Con `pd.melt()` convertimos nombres de columnas en valores de una nueva columna variable:
- `id_vars`: Identificadores a conservar.
- `value_vars`: Columnas a aplanar.
- `var_name`: Nombre de la nueva columna de variables.
- `value_name`: Nombre de la nueva columna de valores.

In [ ]:
df_tidy = pd.melt(
    df_clinico,
    id_vars=["paciente"],
    value_vars=["medicamento_a", "medicamento_b"],
    var_name="medicamento",
    value_name="frecuencia_cardiaca"
)

# Normalizar etiquetas
df_tidy["medicamento"] = df_tidy["medicamento"].str.replace("medicamento_", "").str.upper()
print("--- Tabla Reestructurada (Tidy Data) ---")
display(df_tidy)

## 2. Reestructuración Inversa: `pivot()` y `pivot_table()`

Para generar matrices de correlación o reportes ejecutivos resumidos, revertimos el formato usando `.pivot()`.

In [ ]:
df_ancho_recuperado = df_tidy.pivot(
    index="paciente",
    columns="medicamento",
    values="frecuencia_cardiaca"
).reset_index()

df_ancho_recuperado.columns.name = None
display(df_ancho_recuperado)

## 📝 Ejercicios Prácticos

### Ejercicio 1 (Guiado)
Transforma un reporte financiero trimestral de ventas a formato tidy.

In [ ]:
df_ventas = pd.DataFrame({
    "sucursal": ["Norte", "Sur", "Centro"],
    "Q1": [12000, 15000, 20000],
    "Q2": [13500, 16200, 21500],
    "Q3": [14000, 15800, 19800],
    "Q4": [18000, 22000, 26000]
})

df_ventas_tidy = pd.melt(df_ventas, id_vars=["sucursal"], var_name="trimestre", value_name="ingreso_usd")
display(df_ventas_tidy.head(6))

### Ejercicio 2 (Independiente)
A partir del dataset `df_sensores`, calcula la temperatura promedio agrupada por sensor y fecha.

In [ ]:
df_sensores = pd.DataFrame({
    "sensor_id": ["S1", "S1", "S2", "S2"],
    "fecha": ["2026-05-01", "2026-05-02", "2026-05-01", "2026-05-02"],
    "temp_manana": [20.4, 21.1, 19.5, 20.0],
    "temp_tarde": [28.2, 29.0, 27.1, 28.5]
})

# Solución:
df_sensores_tidy = pd.melt(df_sensores, id_vars=["sensor_id", "fecha"], var_name="momento", value_name="temperatura")
promedio = df_sensores_tidy.groupby(["sensor_id", "fecha"])["temperatura"].mean().reset_index()
display(promedio)

## 📋 Tabla de Métodos y Resumen

| Método | Operación | Cuándo utilizarlo |
|:---|:---|:---|
| `pd.melt()` | Wide a Long | Cuando columnas contienen nombres de categorías |
| `df.pivot()` | Long a Wide | Reorganizar llaves únicas sin colisiones de agregación |
| `df.pivot_table()` | Long a Wide agregada | Cuando existen filas duplicadas para un par clave |
| `df.stack()` / `unstack()` | Manipular MultiIndex | Reordenar jerarquías en índices complejos |

### 💡 Conclusión
El principio Tidy Data garantiza consistencia sintáctica y hace que la aplicación de funciones analíticas, visualizaciones estadísticas y algoritmos de Machine Learning sea directa y eficiente.